In [1]:
!pip install rapidfuzz

In [2]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

In [3]:
df = pd.read_excel("/Users/hotboytraidat/Desktop/Cell_marker_All.xlsx")

for col in ["species", "cell_name", "cell_type", "Symbol"]:
    df[col] = df[col].fillna("").astype(str).str.strip()

# bỏ dòng thiếu species hoặc gene symbol
df = df[
    (df["species"] != "") &
    (df["Symbol"] != "")
].copy()

# tạo nhãn cell để chọn
df["cell_label"] = df["cell_name"]
df.loc[df["cell_label"] == "", "cell_label"] = df.loc[df["cell_label"] == "", "cell_type"]

# bỏ dòng vẫn không có cell label
df = df[df["cell_label"] != ""].copy()

print("Rows:", len(df))
print("Species:", sorted(df["species"].unique().tolist()))
print("Total selectable cells:", df["cell_label"].nunique())
df.head()

Rows: 82887
Species: ['Human', 'Mouse']
Total selectable cells: 2330


,species,tissue_class,tissue_type,uberonongology_id,cancer_type,cell_type,cell_name,cellontology_id,marker,Symbol,...,Genetype,Genename,UNIPROTID,technology_seq,marker_source,PMID,Title,journal,year,cell_label
0,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,MERTK,MERTK,...,protein_coding,"MER proto-oncogene, tyrosine kinase",Q12866,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
1,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD16,FCGR3A,...,protein_coding,Fc fragment of IgG receptor IIIb,O75015,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
2,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD206,MRC1,...,protein_coding,mannose receptor C-type 1,P22897,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
3,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CRIg,VSIG4,...,protein_coding,V-set and immunoglobulin domain containing 4,Q9Y279,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage
4,Human,Abdomen,Abdomen,UBERON_0000916,Normal,Normal cell,Macrophage,CL_0000235,CD163,CD163,...,protein_coding,CD163 molecule,Q86VB7,NaN,Experiment,31982413.0,Peritoneal Level of CD206 Associates With Mort...,Gastroenterology,2020.0,Macrophage


In [4]:
cell_options = sorted(df["cell_label"].unique().tolist())
cell_options[:5000]

['1-cell stage cell (Blastomere)',
 '4-cell stage cell (Blastomere)',
 '8-cell stage cell (Blastomere)',
 'A1 astrocyte',
 'A2 astrocyte',
 'AIRE-enriched cell',
 'AXL+ SIGLEC6+ dendritic cell',
 'AXL+SIGLEC6+ dendritic cell',
 'Abnormal myeloid cell',
 'Abnormal plasma cell',
 'Acinar cell',
 'Activated B cell',
 'Activated CD25+ regulatory T cell',
 'Activated CD4+ T cell',
 'Activated CD8+ T cell',
 'Activated T cell',
 'Activated T helper 1 cell',
 'Activated cell',
 'Activated dendritic cell',
 'Activated double-positive T cell',
 'Activated effector cell',
 'Activated effector memory CD4+ T cell',
 'Activated effector memory CD8+ T cell',
 'Activated endothelial cell',
 'Activated eosinophil',
 'Activated fibroblast',
 'Activated hepatic stellate cell',
 'Activated immune cell',
 'Activated langerhans cell',
 'Activated macrophage',
 'Activated memory B cell',
 'Activated memory CD8+ T cell',
 'Activated memory T cell',
 'Activated microglial cell',
 'Activated myofibroblast',
 '

In [5]:
search_box = widgets.Text(
    value="",
    placeholder="Type here if you cannot find the cell in the dropdown",
    description="Search:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

cell_dropdown = widgets.Dropdown(
    options=cell_options,
    description="Choose cell:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="500px")
)

species_dropdown = widgets.Dropdown(
    options=["Human", "Mouse"],
    value="Human",
    description="Species:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="250px")
)

search_button = widgets.Button(
    description="Find genes",
    button_style="success"
)

output = widgets.Output()

In [6]:
def update_dropdown(change):
    keyword = change["new"].strip().lower()

    if keyword == "":
        filtered = cell_options
    else:
        filtered = [x for x in cell_options if keyword in x.lower()]

    if len(filtered) == 0:
        cell_dropdown.options = ["No match found"]
        cell_dropdown.value = "No match found"
    else:
        cell_dropdown.options = filtered
        cell_dropdown.value = filtered[0]

search_box.observe(update_dropdown, names="value")

In [8]:
def find_genes(selected_cell, selected_species):
    if selected_cell == "No match found":
        return []

    result = df.copy()

    # lọc species trước
    result = result[result["species"].str.lower() == selected_species.lower()]

    # ưu tiên khớp chính xác cell_name
    exact_name = result[result["cell_name"].str.lower() == selected_cell.lower()]

    if len(exact_name) > 0:
        result = exact_name
    else:
        # fallback sang cell_type
        result = result[result["cell_type"].str.lower() == selected_cell.lower()]

    # lấy gene unique theo alphabet
    genes = sorted(result["Symbol"].dropna().unique().tolist())

    return genes

In [9]:
def run_search(b):
    with output:
        clear_output()

        selected_cell = cell_dropdown.value
        selected_species = species_dropdown.value

        genes = find_genes(selected_cell, selected_species)

        if len(genes) == 0:
            print("No genes found.")
        else:
            print("\n".join(genes))

search_button.on_click(run_search)

In [10]:
display(search_box)
display(cell_dropdown)
display(species_dropdown)
display(search_button)
display(output)

Text(value='', description='Search:', layout=Layout(width='500px'), placeholder='Type here if you cannot find …

Dropdown(description='Choose cell:', layout=Layout(width='500px'), options=('1-cell stage cell (Blastomere)', …

Dropdown(description='Species:', layout=Layout(width='250px'), options=('Human', 'Mouse'), style=DescriptionSt…

Button(button_style='success', description='Find genes', style=ButtonStyle())

Output()